# T08 — Protocolo walk-forward comum

Script: `walk_forward.py`. Vale para os 20 pipelines (4 modelos × 5 bases).

- **Corte:** cronológico, sem embaralhamento — Bitcoin 70/30; tráfego, poluição e clima 80/20; ouro 75/25 (`inicio_teste` de `dados_tratados`).
- **Horizonte:** 1 passo (dia, hora ou semana).
- **Origens:** datas do teste com alvo e features completos (`preparar_features_base`); iguais para todos os modelos da base.
- **Causalidade:** features só até `t-1`; escalonadores e hiperparâmetros ajustados só no histórico; teste nunca usado para escolher parâmetros.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'walk_forward.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from walk_forward import BASES, POLITICAS, protocolo, validar_previsoes


## Protocolo por base

In [2]:
tabela_protocolo = pd.DataFrame([protocolo(b, PROJECT_ROOT) for b in BASES])
display(tabela_protocolo)


,base,alvo,frequencia,treino,seed,inicio_teste,horizonte,n_treino,n_origens,n_grade_teste,cobertura_grade_pct,primeira_origem,ultima_origem
0,bitcoin,priceClose,D,0.70,42,2024-04-13 12:00:00,1,1934,884,884,100.000000,2024-04-13 12:00:00,2026-09-13 12:00:00
1,trafego,traffic_volume,h,0.80,67,2017-07-20 01:00:00,1,21327,10267,10511,97.678622,2017-07-20 01:00:00,2018-09-30 23:00:00
2,poluicao,PM2.5,h,0.80,42,2016-05-12 19:00:00,1,23449,6336,7013,90.346499,2016-05-12 19:00:00,2017-02-28 23:00:00
3,clima,T (degC),h,0.80,42,2015-05-27 15:00:00,1,55752,13785,14026,98.281762,2015-05-27 15:00:00,2017-01-01 00:00:00
4,ouro,VALUE,W-FRI,0.75,42,2002-10-11 00:00:00,1,1775,601,601,100.000000,2002-10-11 00:00:00,2014-04-11 00:00:00


## Política de treino por modelo

In [3]:
display(pd.DataFrame(POLITICAS.items(), columns=['modelo', 'politica']))


,modelo,politica
0,random_forest,modelo fixo ajustado no treino; entradas atual...
1,mlp,modelo fixo ajustado no treino; entradas atual...
2,sarimax,reajuste periódico em janela deslizante; estad...
3,holt_winters,"a definir pelo responsável, respeitando corte ..."


## Conferência das previsões em cada notebook

As saídas ficam nos próprios notebooks. Depois que um modelo montar suas previsões, ele deve chamar `validar_previsoes` para conferir que há exatamente uma previsão em cada origem elegível da base. O protocolo não depende de CSVs ou pastas de resultados.

```python
from walk_forward import validar_previsoes

# Exemplo dentro do notebook do modelo, após montar `previsoes`:
conferencia = validar_previsoes(BASE_NAME, previsoes.index, PROJECT_ROOT)
assert conferencia["ok"], conferencia
```

A conferência verifica as **datas**. O modelo continua responsável por provar que cada entrada respeita o horizonte e que seus hiperparâmetros foram escolhidos antes do teste.

## Limitações ainda abertas

- O protocolo define corte, horizonte e origens elegíveis. A política de reajuste do modelo também precisa ser declarada; não é a mesma para RF/MLP e SARIMAX.
- O SARIMAX do Grupo 4 na `main` foi executado com tratamento próprio da fonte e reajustes condicionados ao tempo de máquina. Sua integração à base tratada comum e a reprodução dos resultados são pendências antes do ranking final.
- O Holt-Winters do Grupo 1 ainda está em frequência mensal exploratória, fora do horizonte diário oficial.
- As máscaras comuns deixam de fora períodos com dados incompletos. O relatório deve mostrar cobertura junto ao MAE.